# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guided exploration of the FAIR<sup>2</sup> protein dataset using the `mlcroissant` library, leveraging its Croissant schema metadata and well-defined record sets and fields via their `@id`.

### Dataset Source
The dataset source is defined by a Croissant schema, accessible from:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed (uncomment if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Keep as object - avoid treating as dict/list
print("Dataset Title:", getattr(meta, "name", None))
print("Description:\n", getattr(meta, "description", None))

## 2. Data Overview
Review available record sets, fields, and their IDs, referencing all by their `@id` according to the Croissant schema.

In [ ]:
# List all record sets with @id

recordset_ids = [r['@id'] for r in getattr(meta, 'record_sets', getattr(meta, 'recordSet', []))]

if not recordset_ids:
    # Sometimes the metadata may use 'recordSet' or there may be only a default main record set derived from distribution
    # Let's try loading the distribution as a record set
    dist = getattr(meta, 'distribution', [])
    if dist:
        # The @id of the main record set is usually the @id of the data file, but if not available, we can load and let mlcroissant display them
        for recset in dataset.record_sets:
            print(f"Record Set: {recset['@id']}")
            print(f"  - Name: {recset.get('name')}")
            print(f"  - Description: {recset.get('description')}")
            print(f"  - Fields: ")
            for field in recset.get('fields', []):
                print(f"      - {field['@id']} -- {field.get('name')}")
        recordset_ids = [recset['@id'] for recset in dataset.record_sets]
else:
    for recid in recordset_ids:
        recset = dataset.record_set(recid)
        print(f"Record Set: {recid}")
        print(f"  - Name: {recset.name}")
        print(f"  - Fields:")
        for field in recset.fields:
            print(f"      - {field['@id']} -- {field.name}")

# If we didn't find any, we can discover from dataset.record_sets:
if not recordset_ids:
    recordset_ids = [recset['@id'] for recset in dataset.record_sets]
    print("Found record sets:", recordset_ids)

# Example: Print records from the main record set
if recordset_ids:
    print(f"\nSample record from record set {recordset_ids[0]}")
    for idx, x in enumerate(dataset.records(record_set=recordset_ids[0])):
        print(x)
        if idx > 1:  # Show a few records
            break

## 3. Data Extraction
Load data from the primary record set (`@id`) into a DataFrame for analysis. All fields are referenced by their `@id`.

In [ ]:
# Get all available record set IDs
record_sets = [recset['@id'] for recset in dataset.record_sets]
dataframes = {}

# Load all record sets to pandas
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded record set: {record_set_id} -- shape: {df.shape}")
    print(f"Columns (@id): {df.columns.tolist()}\n")
    dataframes[record_set_id] = df

# For exploration, choose the main record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Columns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process, filter, and analyze the data. We'll select a numeric field by its `@id` (example: protein coverage or molecular weight).

In [ ]:
# Pick a numeric field (by `@id`).
df = dataframes[main_record_set_id].copy()
print(f"Data columns: {df.columns.tolist()}")

# Find a numeric column - commonly these have 'Coverage', 'MW', or 'Abundance' in their schema name/label
numeric_field = None
for c in df.columns:
    if any(lbl in c.lower() for lbl in ["coverage", "mw", "mass", "abundance", "count", "peptide", "score"]):
        numeric_field = c
        break

if numeric_field is None:
    # fallback: pick any numeric-looking column
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break

print(f"Selected numeric field (for EDA): {numeric_field}")

# Drop non-numeric rows, convert to float if necessary
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
print(f"Unique values in {numeric_field}: {df[numeric_field].dropna().unique()[:5]}")

threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
# Filter records
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): Count = {len(filtered_df)}")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical field (e.g., sample, description, label, accession etc.)
group_field = None
for c in df.columns:
    if c != numeric_field and df[c].nunique() < min(20, len(df)//3):
        group_field = c
        break

print(f"Group field: {group_field}")

if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean {numeric_field} values by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationships by groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color="skyblue")
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field available, boxplot by groups
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df, palette="Set2")
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We successfully loaded the FAIR<sup>2</sup> protein mass spectrometry dataset and explored its composition using the `mlcroissant` library, referencing schema entities by their unique `@id`.
- We examined the available record sets and fields, loaded records, and performed basic filtering and normalization on representative numeric fields.
- Visualizations of field distributions and group comparisons offer insight into the data's structure and potential for downstream biological or statistical analysis.

Next steps could include more advanced feature engineering, deeper exploratory analysis, or application of machine learning models as relevant.